In [0]:
%sql
CREATE OR REPLACE TABLE dev_gold.common.ont_entities (
  entity_name    STRING NOT NULL COMMENT 'Business name of the entity',
  description    STRING COMMENT 'What the entity means in business terms',
  physical_table STRING COMMENT 'Gold table or view that holds it',
  primary_key    STRING COMMENT 'Column(s) that uniquely identify one row',
  grain          STRING COMMENT 'What one row represents',
  CONSTRAINT pk_ont_entities PRIMARY KEY (entity_name)
) COMMENT 'Ontology: business entities and where they are stored';

INSERT OVERWRITE dev_gold.common.ont_entities VALUES
 ('Customer',   'A company that buys products from us',            'dev_gold.common.vw_customers_all',   'customer_id',             'one row per customer'),
 ('Order',      'A purchase placed by a customer on a date',       'dev_gold.common.vw_orders_all',      'order_id',                'one row per order'),
 ('Order Line', 'One product within an order; holds revenue and quantity', 'dev_gold.common.vw_order_lines_all', 'order_id, line_number', 'one row per product per order'),
 ('Product',    'An item we sell',                                 'dev_gold.common.dim_product',        'product_id',              'one row per product'),
 ('Supplier',   'A company that supplies the products we sell',    'dev_gold.common.dim_supplier',       'supplier_id',             'one row per supplier'),
 ('Geography',  'A country and the world region it belongs to',    'dev_gold.common.dim_geography',      'nation_id',               'one row per country'),
 ('Date',       'A calendar day',                                  'dev_gold.common.dim_date',           'date_key',                'one row per day');

In [0]:
%sql
CREATE OR REPLACE TABLE dev_gold.common.ont_relationships (
  from_entity    STRING NOT NULL,
  relationship   STRING NOT NULL COMMENT 'Business verb, e.g. places, contains',
  to_entity      STRING NOT NULL,
  cardinality    STRING COMMENT '1:N or N:1',
  join_condition STRING COMMENT 'Exact SQL join between the physical tables',
  notes          STRING
) COMMENT 'Ontology: relationships between entities and how to join them';

INSERT OVERWRITE dev_gold.common.ont_relationships VALUES
 ('Customer',   'places',            'Order',      '1:N', 'vw_orders_all.customer_id = vw_customers_all.customer_id', NULL),
 ('Order',      'contains',          'Order Line', '1:N', 'vw_order_lines_all.order_id = vw_orders_all.order_id', NULL),
 ('Order Line', 'is for',            'Product',    'N:1', 'vw_order_lines_all.product_id = dim_product.product_id', NULL),
 ('Order Line', 'is supplied by',    'Supplier',   'N:1', 'vw_order_lines_all.supplier_id = dim_supplier.supplier_id', NULL),
 ('Customer',   'is located in',     'Geography',  'N:1', 'vw_customers_all.nation_id = dim_geography.nation_id', 'Use for CUSTOMER country/region'),
 ('Supplier',   'is located in',     'Geography',  'N:1', 'dim_supplier.nation_id = dim_geography.nation_id', 'Use for SUPPLIER country/region'),
 ('Order',      'was placed on',     'Date',       'N:1', 'vw_orders_all.order_date_key = dim_date.date_key', 'Use for order counts and order trends'),
 ('Order Line', 'was shipped on',    'Date',       'N:1', 'vw_order_lines_all.ship_date_key = dim_date.date_key', 'Use for revenue by period (revenue is recognised on ship date)');

In [0]:
%sql
CREATE OR REPLACE TABLE dev_gold.common.ont_business_glossary (
  term           STRING NOT NULL COMMENT 'Official business term',
  term_type      STRING COMMENT 'metric, dimension or filter',
  definition     STRING COMMENT 'Plain-English meaning',
  synonyms       ARRAY<STRING> COMMENT 'Other words users say for this term',
  sql_expression STRING COMMENT 'How to calculate it in SQL',
  source_object  STRING COMMENT 'Table/view the expression applies to',
  CONSTRAINT pk_glossary PRIMARY KEY (term)
) COMMENT 'Ontology: business glossary mapping business terms to SQL';

INSERT OVERWRITE dev_gold.common.ont_business_glossary VALUES
 ('Net Revenue',        'metric',    'Revenue after discount. Default meaning of "revenue" or "sales".',
    array('revenue','sales','net sales','turnover','income'),            'SUM(net_amount)',                                   'vw_order_lines_all'),
 ('Gross Revenue',      'metric',    'Revenue before discount',
    array('gross sales','list revenue'),                                 'SUM(gross_amount)',                                 'vw_order_lines_all'),
 ('Discount Amount',    'metric',    'Money given away as discount',
    array('discounts','markdown'),                                       'SUM(gross_amount - net_amount)',                    'vw_order_lines_all'),
 ('Units Sold',         'metric',    'Total quantity of items sold',
    array('quantity','volume','units'),                                  'SUM(quantity)',                                     'vw_order_lines_all'),
 ('Order Count',        'metric',    'Number of distinct orders',
    array('number of orders','orders','order volume'),                   'COUNT(DISTINCT order_id)',                          'vw_orders_all'),
 ('Average Order Value','metric',    'Net revenue divided by number of orders',
    array('AOV','average order size','basket size'),                     'SUM(net_amount) / COUNT(DISTINCT order_id)',        'vw_order_lines_all'),
 ('Return Rate',        'metric',    'Share of order lines that were returned',
    array('returns percentage','return ratio'),                          "AVG(CASE WHEN return_flag = 'R' THEN 1.0 ELSE 0.0 END)", 'vw_order_lines_all'),
 ('Sales Region',       'dimension', 'World region of the CUSTOMER who placed the order',
    array('region','territory','market area'),                           'sales_region',                                      'vw_*_all'),
 ('Customer Country',   'dimension', 'Country of the customer',
    array('country','nation','market'),                                  'dim_geography.nation_name (joined via customer nation_id)', 'dim_geography'),
 ('Market Segment',     'dimension', 'Industry the customer belongs to',
    array('industry','segment','vertical','sector'),                     'market_segment',                                    'vw_customers_all'),
 ('Brand',              'dimension', 'Product brand',
    array('label','make'),                                               'brand',                                             'dim_product'),
 ('Open Order',         'filter',    'Order not yet fulfilled',
    array('pending order','unfulfilled order','backlog'),                "order_status = 'Open'",                             'vw_orders_all'),
 ('Returned Item',      'filter',    'Order line that the customer returned',
    array('return','returned'),                                          "return_flag = 'R'",                                 'vw_order_lines_all'),
 ('Revenue Period',     'filter',    'Revenue is recognised on SHIP date; order counts use ORDER date',
    array('when sold','sales date'),                                     'join dim_date on ship_date_key for revenue; on order_date_key for orders', 'dim_date');

In [0]:
%sql
CREATE OR REPLACE TABLE dev_gold.common.ont_value_synonyms (
  column_name     STRING NOT NULL COMMENT 'Column the value belongs to',
  canonical_value STRING NOT NULL COMMENT 'Exact value stored in the data',
  synonyms        ARRAY<STRING>   COMMENT 'Words users might use instead'
) COMMENT 'Ontology: maps user words to exact stored values for filters';

INSERT OVERWRITE dev_gold.common.ont_value_synonyms VALUES
 ('market_segment', 'AUTOMOBILE',          array('automotive','auto','car makers','cars')),
 ('market_segment', 'BUILDING',            array('construction','builders')),
 ('market_segment', 'FURNITURE',           array('furnishings','furniture makers')),
 ('market_segment', 'HOUSEHOLD',           array('home','consumer goods')),
 ('market_segment', 'MACHINERY',           array('industrial','machines','equipment')),
 ('sales_region',   'AMERICA',             array('americas','north and south america')),
 ('sales_region',   'MIDDLE EAST',         array('middle east','gulf','mideast')),
 ('order_status',   'Open',                array('pending','not shipped','unfulfilled')),
 ('order_status',   'Fulfilled',           array('completed','closed','delivered')),
 ('order_status',   'Partially fulfilled', array('partial','partly shipped')),
 ('return_flag',    'R',                   array('returned')),
 ('return_flag',    'A',                   array('accepted','kept')),
 ('ship_mode',      'AIR',                 array('air freight','by air','flight')),
 ('ship_mode',      'TRUCK',               array('road','lorry')),
 ('order_priority', '1-URGENT',            array('urgent','critical','top priority'));

In [0]:
%sql
CREATE OR REPLACE VIEW dev_gold.common.vw_sales_enriched (
  sales_region      COMMENT 'Region of the customer. Also called: region, territory',
  order_id          COMMENT 'Order ID',
  line_number       COMMENT 'Line within the order',
  order_date        COMMENT 'Date the order was placed',
  order_status      COMMENT 'Fulfilled, Open or Partially fulfilled. Open = pending/backlog',
  order_priority    COMMENT '1-URGENT to 5-LOW',
  customer_id       COMMENT 'Customer ID',
  customer_name     COMMENT 'Customer name',
  market_segment    COMMENT 'Customer industry: AUTOMOBILE, BUILDING, FURNITURE, HOUSEHOLD, MACHINERY. Also called: industry, segment, vertical',
  customer_country  COMMENT 'Country of the customer. Also called: country, nation, market',
  product_id        COMMENT 'Product ID',
  brand             COMMENT 'Product brand, e.g. Brand#13',
  product_type      COMMENT 'Product type, e.g. PROMO BRUSHED COPPER',
  manufacturer      COMMENT 'Product manufacturer',
  supplier_name     COMMENT 'Supplier of the product',
  supplier_country  COMMENT 'Country of the supplier (not the customer)',
  ship_date         COMMENT 'Date shipped. Revenue is recognised on this date',
  ship_year         COMMENT 'Year of ship date. Use for revenue by year',
  ship_quarter      COMMENT 'Quarter of ship date (1-4)',
  ship_month        COMMENT 'Month number of ship date (1-12)',
  ship_mode         COMMENT 'AIR, RAIL, SHIP, TRUCK, MAIL, FOB, REG AIR',
  quantity          COMMENT 'Units sold. Also called: volume, units',
  gross_amount      COMMENT 'Revenue before discount (USD). Also called: gross sales',
  discount_pct      COMMENT 'Discount fraction, 0.05 = 5 percent',
  net_amount        COMMENT 'Revenue after discount (USD). DEFAULT meaning of revenue, sales, turnover',
  return_flag       COMMENT 'R = returned, A = accepted, N = not decided'
)
COMMENT 'Business-friendly sales view: one row per order line with customer, product, supplier, geography and date attributes. Preferred source for text-to-SQL.'
AS
SELECT l.sales_region, l.order_id, l.line_number,
       o.order_date, o.order_status, o.order_priority,
       c.customer_id, c.customer_name, c.market_segment, cg.nation_name,
       p.product_id, p.brand, p.product_type, p.manufacturer,
       s.supplier_name, sg.nation_name,
       l.ship_date, d.year, d.quarter, d.month_num, l.ship_mode,
       l.quantity, l.gross_amount, l.discount_pct, l.net_amount, l.return_flag
FROM dev_gold.common.vw_order_lines_all l
JOIN dev_gold.common.vw_orders_all    o  ON l.order_id    = o.order_id
JOIN dev_gold.common.vw_customers_all c  ON o.customer_id = c.customer_id
JOIN dev_gold.common.dim_geography    cg ON c.nation_id   = cg.nation_id
JOIN dev_gold.common.dim_product      p  ON l.product_id  = p.product_id
JOIN dev_gold.common.dim_supplier     s  ON l.supplier_id = s.supplier_id
JOIN dev_gold.common.dim_geography    sg ON s.nation_id   = sg.nation_id
JOIN dev_gold.common.dim_date         d  ON l.ship_date_key = d.date_key;

In [0]:
%sql
 select * from dev_gold.common.vw_sales_enriched

In [0]:
%sql
CREATE OR REPLACE VIEW dev_gold.common.mv_sales
WITH METRICS
LANGUAGE YAML
COMMENT 'Certified sales metrics. Use MEASURE(<name>) to query.'
AS $$
version: 0.1
source: dev_gold.common.vw_sales_enriched
dimensions:
  - name: sales_region
    expr: sales_region
  - name: customer_country
    expr: customer_country
  - name: market_segment
    expr: market_segment
  - name: brand
    expr: brand
  - name: product_type
    expr: product_type
  - name: supplier_country
    expr: supplier_country
  - name: ship_mode
    expr: ship_mode
  - name: order_status
    expr: order_status
  - name: ship_year
    expr: ship_year
  - name: ship_month
    expr: DATE_TRUNC('MONTH', ship_date)
measures:
  - name: net_revenue
    expr: SUM(net_amount)
  - name: gross_revenue
    expr: SUM(gross_amount)
  - name: discount_amount
    expr: SUM(gross_amount - net_amount)
  - name: units_sold
    expr: SUM(quantity)
  - name: order_count
    expr: COUNT(DISTINCT order_id)
  - name: customer_count
    expr: COUNT(DISTINCT customer_id)
  - name: average_order_value
    expr: SUM(net_amount) / COUNT(DISTINCT order_id)
  - name: return_rate
    expr: AVG(CASE WHEN return_flag = 'R' THEN 1.0 ELSE 0.0 END)
$$;




In [0]:
%sql
SELECT ship_year, sales_region,
       MEASURE(net_revenue)         AS net_revenue,
       MEASURE(order_count)         AS orders,
       MEASURE(average_order_value) AS aov
FROM dev_gold.common.mv_sales
GROUP BY ALL
ORDER BY ship_year, sales_region;

In [0]:
G, regions = "dev_gold.common", ["africa", "america", "asia", "europe", "middle_east"]
# Table / view level: which ontology entity each object represents
entity_tags = {
    f"{G}.dim_date": "Date", f"{G}.dim_geography": "Geography",
    f"{G}.dim_product": "Product", f"{G}.dim_supplier": "Supplier",
}
for obj, ent in entity_tags.items():
    spark.sql(f"ALTER TABLE {obj} SET TAGS ('domain'='sales', 'layer'='gold', 'ontology_entity'='{ent}')")
for v, ent in {"vw_customers_all": "Customer", "vw_orders_all": "Order",
               "vw_order_lines_all": "Order Line", "vw_sales_enriched": "Order Line"}.items():
    spark.sql(f"ALTER VIEW {G}.{v} SET TAGS ('domain'='sales', 'layer'='gold', 'ontology_entity'='{ent}')")
    spark.sql(f"ALTER VIEW {G}.mv_sales SET TAGS ('domain'='sales', 'certified'='true', 'object_type'='metric_view')")
def tag_region(r):
    S = f"dev_gold.{r}"
    spark.sql(f"ALTER TABLE {S}.dim_customer SET TAGS ('domain'='sales', 'ontology_entity'='Customer')")
    spark.sql(f"ALTER TABLE {S}.fact_orders SET TAGS ('domain'='sales', 'ontology_entity'='Order')")
    spark.sql(f"ALTER TABLE {S}.fact_order_lines SET TAGS ('domain'='sales', 'ontology_entity'='Order Line')")
    # Column level: business terms and sensitive data
    spark.sql(f"ALTER TABLE {S}.fact_order_lines ALTER COLUMN net_amount SET TAGS ('business_term'='Net Revenue')")
    spark.sql(f"ALTER TABLE {S}.fact_order_lines ALTER COLUMN gross_amount SET TAGS ('business_term'='Gross Revenue')")
    spark.sql(f"ALTER TABLE {S}.fact_order_lines ALTER COLUMN quantity SET TAGS ('business_term'='Units Sold')")
    spark.sql(f"ALTER TABLE {S}.dim_customer ALTER COLUMN phone SET TAGS ('pii'='phone')")
    print("✔ tagged", r)
tag_region("africa")
tag_region("america")
tag_region("asia")
tag_region("europe")
tag_region("middle_east")




In [0]:
%sql
-- Everything the ontology contains, in one place
SELECT 'entities' AS part, COUNT(*) AS items FROM dev_gold.common.ont_entities
UNION ALL SELECT 'relationships', COUNT(*) FROM dev_gold.common.ont_relationships
UNION ALL SELECT 'glossary terms', COUNT(*) FROM dev_gold.common.ont_business_glossary
UNION ALL SELECT 'value synonyms', COUNT(*) FROM dev_gold.common.ont_value_synonyms;




In [0]:
%sql
-- Which objects carry which tags
SELECT schema_name, table_name, tag_name, tag_value
FROM dev_gold.information_schema.table_tags
ORDER BY 1, 2;

In [0]:
%sql
-- The glossary as text, ready to paste into Genie instructions in Step 6
SELECT concat_ws('\n', collect_list(
         concat('- ', term, ' (also: ', array_join(synonyms, ', '), '): ',
                definition, '. SQL: ', sql_expression))) AS genie_instructions
FROM dev_gold.common.ont_business_glossary;

In [0]:
%sql
 select * from dev_gold.common.ont_entities

In [0]:
%sql
-- The glossary as text, ready to paste into Genie instructions in Step 6
SELECT concat_ws('\n', collect_list(
         concat('- ', term, ' (also: ', array_join(synonyms, ', '), '): ',
                definition, '. SQL: ', sql_expression))) AS genie_instructions
FROM dev_gold.common.ont_business_glossary;
